# 04 — XGBoost Forecasting Model

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np

## 1. Load data

In [2]:
featured = pd.read_csv('../data/processed/features.csv', parse_dates=['date'])
from src.features import get_feature_columns
FEATURE_COLS = [c for c in get_feature_columns() if c in featured.columns]
TARGET = 'new_cases_7d'

cutoff = featured['date'].max() - pd.Timedelta(days=30)
train = featured[featured.date <= cutoff]
test  = featured[featured.date >  cutoff]

X_train = train[FEATURE_COLS].fillna(0).values
y_train = train[TARGET].fillna(0).values
X_test  = test[FEATURE_COLS].fillna(0).values
y_test  = test[TARGET].fillna(0).values

## 2. Train XGBoost

In [4]:
!pip install xgboost


  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)


In [5]:
from src.models.xgboost_model import XGBForecaster

model = XGBForecaster(n_estimators=500, max_depth=5)
model.fit(X_train, y_train,
          feature_names=FEATURE_COLS,
          eval_set=(X_test, y_test))

[0]	validation_0-rmse:2.97339
[50]	validation_0-rmse:0.28176
[100]	validation_0-rmse:0.12199
[150]	validation_0-rmse:0.11162
[200]	validation_0-rmse:0.10654
[250]	validation_0-rmse:0.10109
[300]	validation_0-rmse:0.09679
[350]	validation_0-rmse:0.09433
[400]	validation_0-rmse:0.09206
[450]	validation_0-rmse:0.09042
[499]	validation_0-rmse:0.08939
  XGB best iteration: 499


## 3. Evaluate

In [6]:
from src.evaluation import evaluate
preds = model.predict(X_test)
metrics = evaluate(y_test, preds, name='XGBoost')

  [XGBoost]  RMSE=173.3  MAE=29.9  MAPE=4.5%  R²=0.9976


## 4. Feature importance & SHAP

In [8]:
!pip install shap


  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/547.0 kB ? eta -:--:--
   ---------------------------------------- 547.0/547.0 kB 6.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 6.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   - -------------------------------------- 1.3/38.1 MB 6.7 MB/s eta 0:00:06
   -- ------------------------------------- 2.6/38.1 MB 6.6 MB/s eta 0:00:06
   --- ------------------------------------ 3.7/38.1 MB 5.7 MB/s eta 0:00:07
   ----- ---------------------------------- 5.0/38.1 MB 5.8 MB/s eta 0:00:06
   ------ --------------------------------- 6.6/38.1 MB 6.1 MB/s eta 0:00:06
   -------- ------------------------------- 7.9/38.1 MB 6.2 MB/s eta 0:00:05
   --------- -----------------------------

In [11]:
# model.plot_feature_importance(top_n=15)
# shap_vals = model.plot_shap(X_test, FEATURE_COLS)
def plot_feature_importance(self, top_n=15):
    import matplotlib.pyplot as plt
    import numpy as np
    import os

    os.makedirs(PLOT_DIR, exist_ok=True)
    booster = self.model.get_booster()
    importance = booster.get_score(importance_type='weight')

    # Sort features by importance
    sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:top_n]
    names, scores = zip(*sorted_importance)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(names, scores)
    ax.set_xlabel("Feature Importance (weight)")
    ax.set_ylabel("Features")
    ax.set_title("Top Feature Importances")
    plt.tight_layout()
    plt.show()


def plot_shap(self, X, feature_names=None, max_display=15):
    import shap, os, matplotlib.pyplot as plt

    os.makedirs(PLOT_DIR, exist_ok=True)
    names = feature_names or self.feature_names or [f'f{i}' for i in range(X.shape[1])]

    # Get booster safely
    booster = self.model.get_booster()

    # Fix base_score if stored as string
    try:
        base_score = booster.attr("base_score")
        if base_score and not base_score.replace('.', '').isdigit():
            # Reset to a numeric default
            booster.set_param({"base_score": "0.5"})
    except Exception:
        pass

    # Use TreeExplainer
    explainer = shap.TreeExplainer(booster)
    shap_values = explainer.shap_values(X)

    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X, feature_names=names, max_display=max_display, show=False)
    plt.tight_layout()
    plt.show()

    return shap_values



## 5. Save

In [13]:
model.save('../models/saved/xgb.pkl')

  XGB saved → ../models/saved/xgb.pkl
